# 01 — Manifold Reconstruction and Eigenspectrum Gate

**v1.1 PU Manifold Curvature.** This notebook is a committed deliverable (shipped WITH
outputs, from a deliberate Restart-and-Run-All — see §0.4). Phase 1 owns §0-§5 of this
file; Phase 2 appends §6 onward, described further in §0.5.

Phase 1 (this plan, 01-01) writes §0-§1: environment/reproducibility, dependency install,
and a permanent smoke-config end-to-end self-test proving every layer of the pipeline
(HF config load → seeded subsample → row-alignment assert → L2 normalize → npz cache →
npz read-back → connectivity check → Isomap fit → joblib cache → joblib read-back) works
before the ~1 GB analysis artifact is ever built.

## §0. Environment & Reproducibility

### §0.1 Python floor

In [1]:
import sys
from pathlib import Path

# Make the notebook-local pu_manifold package importable regardless of how the kernel
# was started (D-01 key link: plain relative import, never installed, never imported
# from src/effdim/).
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

assert sys.version_info >= (3, 11), (
    "This notebook requires Python >= 3.11. pyproject.toml declares >=3.8, which is "
    "stale core-dependency drift tracked as LIB-03: scikit-learn 1.9.0 and the "
    "torch/datasets wheels this notebook installs are the real constraint, not the "
    "declared floor. If you are running on Python 3.10 or earlier, create a separate "
    "notebook kernel for this milestone rather than downgrading the pins in "
    "requirements-notebooks.txt."
)
print(f"Python {sys.version.split()[0]} OK (>= 3.11 required)")

Python 3.14.6 OK (>= 3.11 required)


### §0.2 Dependencies

No `## Package Legitimacy Audit` table exists anywhere under `.planning/` (verified
during planning: `grep -rn "Package Legitimacy" .planning/` returns nothing). Per the
documented fallback policy, every package this phase installs is therefore treated as
`[ASSUMED]` and required human confirmation on the registry before install — this was
Task 1 of the Phase 1 plan (01-01), a `checkpoint:human-verify` with
`gate="blocking-human"` that is **never** auto-approvable. Task 1 was reviewed and
**approved**: `torch==2.13.0+cpu` (PyTorch, `+cpu` wheel from
`https://download.pytorch.org/whl/cpu`), `datasets==5.0.1` (Hugging Face `datasets`),
and `matplotlib==3.11.1` were each confirmed as the legitimate, expected project on
PyPI before this cell was authored or run. `numpy`/`scipy`/`scikit-learn`/`faiss-cpu`
are already core `effdim` dependencies and are deliberately not re-pinned here;
`huggingface_hub`/`hf_xet`/`pyarrow` arrive transitively via `datasets`.

In [2]:
%pip install -q -r requirements-notebooks.txt

Note: you may need to restart the kernel to use updated packages.


### §0.3 Reproducibility header

In [3]:
import subprocess

import numpy as np
import scipy
import sklearn
import faiss
import datasets
import torch

# Defined once here (needed for this header) and restated verbatim in §1.1 so that
# section is self-contained for a reader who jumps straight to §1.
SEED = 20260729

git_sha = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()

print("=== Reproducibility header ===")
print(f"SEED           = {SEED}")
print(f"numpy          = {np.__version__}")
print(f"scipy          = {scipy.__version__}")
print(f"scikit-learn   = {sklearn.__version__}")
print(f"faiss          = {faiss.__version__}")
print(f"datasets       = {datasets.__version__}")
print(f"torch          = {torch.__version__}")
print(f"git commit SHA = {git_sha}")

=== Reproducibility header ===
SEED           = 20260729
numpy          = 2.5.1
scipy          = 1.18.0
scikit-learn   = 1.9.0
faiss          = 1.14.3
datasets       = 5.0.1
torch          = 2.13.0+cpu
git commit SHA = 6dedb7f


The git commit SHA is printed for provenance but is **deliberately NOT** part of any
cache key (D-14) — a docstring-only commit must not invalidate a ~1 GB Isomap artifact.
`sklearn`/`numpy`/`scipy` versions, by contrast, **are** part of the fit cache key,
because a library upgrade can silently change numerical results.

### §0.4 Output hygiene policy

1. This notebook is committed **with outputs intact**, produced by a deliberate
   Restart-and-Run-All. It is not run through `nbconvert --clear-output`.
2. `nbstripout` is not installed and no `.gitattributes` filter is added. This repo has
   no pre-commit framework, and adding one for a milestone this size is unwarranted
   ceremony; `nbstripout` is the documented later upgrade path *if* output-diff noise
   ever becomes a real problem, not something adopted pre-emptively.
3. No cell output may embed a bulk array. Outputs here are limited to plots, tables, and
   scalar prints — never a repr of the 10000x768 embedding arrays and never the fitted
   `Isomap` object — so the notebook JSON stays small enough to diff.
4. `.ipynb_checkpoints/` and `notebooks/.cache/` are already present in `.gitignore`
   (verified), so no gitignore change is needed for this phase.

Phase 2, which appends to this same file, inherits this policy rather than re-deciding
it.

### §0.5 Section numbering

Phase 1 owns `§0`-`§5` of this notebook; Phase 2 appends `§6` onward. Existing
sections are never renumbered once written. Subsections use `### §N.M Title`. This
convention, together with D-01's three fixed notebook filenames, is **costly** to change
later: every cache path, cross-notebook check, and doc reference in this milestone is
written against it.

## §1. Smoke-Config End-to-End Self-Test

This section is a **permanent** self-test, not scaffolding to be deleted later: it
exercises every layer this phase touches — HF config load, seeded subsample, row-
alignment assert, L2 normalization, npz cache, npz read-back identity, connectivity
check, deterministic Isomap fit, joblib cache, joblib read-back identity — at a cheap
smoke size (`n_rows=500`) in seconds, before the ~10 minute, ~1 GB analysis fit is ever
paid for.

### §1.1 Configs

In [4]:
# Restated from §0.3 (same value) so this section is self-contained.
SEED = 20260729

SMOKE_CFG = {
    "dataset": "legacysurvey_dinov3_vitb16",
    "seed": SEED,
    "n_rows": 500,
    "normalize": True,
    "n_neighbors": 10,
    "n_components": 4,
    "eigen_solver": "dense",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}

ANALYSIS_CFG = {
    "dataset": "legacysurvey_dinov3_vitb16",
    "seed": SEED,
    "n_rows": 10_000,
    "normalize": True,
    "n_neighbors": None,  # derived by a later plan's connectivity sweep (ISO-01/ISO-02)
    "n_components": None,  # derived in a later plan via the D-12 ceil(median(...)) rule
    "eigen_solver": "dense",
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
}

print("SMOKE_CFG:   ", SMOKE_CFG)
print("ANALYSIS_CFG:", ANALYSIS_CFG)

SMOKE_CFG:    {'dataset': 'legacysurvey_dinov3_vitb16', 'seed': 20260729, 'n_rows': 500, 'normalize': True, 'n_neighbors': 10, 'n_components': 4, 'eigen_solver': 'dense', 'sklearn_version': '1.9.0', 'numpy_version': '2.5.1', 'scipy_version': '1.18.0'}
ANALYSIS_CFG: {'dataset': 'legacysurvey_dinov3_vitb16', 'seed': 20260729, 'n_rows': 10000, 'normalize': True, 'n_neighbors': None, 'n_components': None, 'eigen_solver': 'dense', 'sklearn_version': '1.9.0', 'numpy_version': '2.5.1', 'scipy_version': '1.18.0'}


`SMOKE_CFG["n_components"] = 4` is a smoke-test constant only — it is **not** the
analysis `n_components`, which a later plan derives from the D-12 rule
(`ceil(median(...))` over the 8 geometric/intrinsic `effdim.compute_dim` keys). Because
`ANALYSIS_CFG` differs from `SMOKE_CFG` in `n_rows` (and, once derived, `n_components`),
the two configs produce different `config_key` hashes by construction — itself the
ISO-05 "a config change produces a new cache key" property, demonstrated concretely in
§1.3 below rather than merely asserted in prose.

### §1.2 Subsample and alignment (smoke)

In [5]:
from pu_manifold import load_subsample, assert_alignment, config_key, joblib_cache

smoke_data = load_subsample(SMOKE_CFG)
alignment_stats = assert_alignment(
    smoke_data["hsc"],
    smoke_data["legacysurvey"],
    smoke_data["row_indices"],
    seed=SEED,
)

print("hsc shape:         ", smoke_data["hsc"].shape)
print("legacysurvey shape:", smoke_data["legacysurvey"].shape)
print("row_indices shape: ", smoke_data["row_indices"].shape)
print("alignment stats:   ", alignment_stats)

hsc shape:          (500, 768)
legacysurvey shape: (500, 768)
row_indices shape:  (500,)
alignment stats:    {'s_true': 0.8455923572483339, 'mu_perm': 0.7171236552581232, 'sd_perm': 0.0030748871782488546, 'z': 41.779972578822736, 'margin_z': 5.0, 'n_permutations': 50, 'row_indices_sha256': '3c22a0439ed2838d364e1d1715afd1f815b5cb7367e40450a95822e09239e035'}


### §1.3 Cache round-trip (smoke)

In [6]:
smoke_data_2 = load_subsample(SMOKE_CFG)
for key in smoke_data:
    assert np.array_equal(smoke_data[key], smoke_data_2[key]), (
        f"{key} differs between the two load_subsample(SMOKE_CFG) calls -- cache "
        f"round-trip is not bit-identical."
    )
print("CACHE HIT: second load_subsample(SMOKE_CFG) call returned bit-identical arrays")

smoke_cfg_changed = dict(SMOKE_CFG, n_rows=SMOKE_CFG["n_rows"] + 1)
key_before = config_key(SMOKE_CFG)
key_after = config_key(smoke_cfg_changed)
print(f"config_key(SMOKE_CFG)               = {key_before}")
print(f"config_key(SMOKE_CFG with n_rows+1) = {key_after}")
assert key_before != key_after, "changing n_rows must change the cache key (ISO-05)"
print("ISO-05: a single config field change demonstrably produces a new cache key")

CACHE HIT: second load_subsample(SMOKE_CFG) call returned bit-identical arrays
config_key(SMOKE_CFG)               = a42e1c3087ee5736
config_key(SMOKE_CFG with n_rows+1) = bf4f9657c2b208ed
ISO-05: a single config field change demonstrably produces a new cache key


### §1.4 Connectivity and Isomap fit (smoke)

In [7]:
from scipy.sparse.csgraph import connected_components
from sklearn.manifold import Isomap
from sklearn.neighbors import kneighbors_graph

X_smoke = smoke_data["legacysurvey"]

smoke_graph = kneighbors_graph(
    X_smoke, n_neighbors=SMOKE_CFG["n_neighbors"], mode="distance"
)
n_components_graph, _ = connected_components(smoke_graph, directed=False)
print(f"connected components (smoke, k={SMOKE_CFG['n_neighbors']}): {n_components_graph}")
assert n_components_graph == 1, (
    f"Smoke k-NN graph has {n_components_graph} connected components, not 1. "
    f"sklearn.manifold.Isomap does not raise on a disconnected graph -- it silently "
    f"bridges components with fabricated long edges (PITFALLS Pitfall 1) -- so this "
    f"check must halt rather than let the fit proceed."
)


def _fit_smoke_isomap():
    model = Isomap(
        n_neighbors=SMOKE_CFG["n_neighbors"],
        n_components=SMOKE_CFG["n_components"],
        eigen_solver="dense",
        n_jobs=-1,
    )
    model.fit(X_smoke)
    return model


# eigen_solver="dense" is pinned explicitly here and everywhere in this milestone:
# Isomap has no random_state, and "auto"/"arpack" uses ARPACK with a random start
# vector, so "dense" is what makes the fit deterministic LAPACK (D-15).
smoke_fit_key = config_key(SMOKE_CFG)
isomap_smoke = joblib_cache(f"isomap_{smoke_fit_key}", SMOKE_CFG, _fit_smoke_isomap)

print("dist_matrix_ shape:", isomap_smoke.dist_matrix_.shape)
print("embedding_ shape:  ", isomap_smoke.embedding_.shape)

connected components (smoke, k=10): 1
dist_matrix_ shape: (500, 500)
embedding_ shape:   (500, 4)


The full classical-MDS eigenspectrum audit and the PASS/MARGINAL/FAIL gate
(SPEC-01 through SPEC-07) are **Phase 2's** deliverable and are deliberately absent from
this notebook's §0-§5.

### §1.5 Joblib read-back (smoke)

In [8]:
isomap_smoke_reloaded = joblib_cache(f"isomap_{smoke_fit_key}", SMOKE_CFG, _fit_smoke_isomap)

assert isomap_smoke_reloaded.dist_matrix_.shape == (
    SMOKE_CFG["n_rows"],
    SMOKE_CFG["n_rows"],
), "dist_matrix_ shape does not match the smoke n_rows x n_rows expectation"
assert np.array_equal(
    isomap_smoke_reloaded.embedding_, isomap_smoke.embedding_
), "embedding_ differs between the fresh fit and the cached joblib reload"

print(f"dist_matrix_ shape (reloaded): {isomap_smoke_reloaded.dist_matrix_.shape}")
print("CACHE HIT: joblib_cache reload returned a bit-identical embedding_")

dist_matrix_ shape (reloaded): (500, 500)
CACHE HIT: joblib_cache reload returned a bit-identical embedding_


`notebooks/.cache/` contents are **trusted-local-only**. `joblib.load` is pickle
deserialization (threat T-01-01) — a `.joblib` file obtained from any third party must
never be placed in that directory. `joblib_cache` only ever loads a path it composed
itself from `CACHE_DIR` plus a `config_key` it just computed; there is no helper anywhere
in `pu_manifold` that loads a caller-supplied absolute path.

### §1.6 Analysis subsample (n = 10,000)

This is the real ~60 MB analysis artifact: `subsample_{seed}_{subsample_key}.npz`, the
cached input every subsequent plan in this phase (and Phase 3's decoder target, Phase 4's
MKNN input) reads from. `ANALYSIS_CFG["n_rows"]` was already set to `10_000` in §1.1 -- this
cell asserts that rather than reassigning it, then draws the subsample for the first time.

In [9]:
assert ANALYSIS_CFG["n_rows"] == 10_000, (
    f"ANALYSIS_CFG['n_rows'] = {ANALYSIS_CFG['n_rows']}, expected 10_000."
)

analysis_data = load_subsample(ANALYSIS_CFG)
HSC = analysis_data["hsc"]
LS = analysis_data["legacysurvey"]
HSC_NORMS = analysis_data["hsc_norms"]
LS_NORMS = analysis_data["ls_norms"]
ROW_INDICES = analysis_data["row_indices"]

print("=== Analysis subsample shapes (n=10,000) ===")
print("HSC shape:        ", HSC.shape)
print("LS shape:         ", LS.shape)
print("HSC_NORMS shape:  ", HSC_NORMS.shape)
print("LS_NORMS shape:   ", LS_NORMS.shape)
print("ROW_INDICES shape:", ROW_INDICES.shape)

assert HSC.shape == (10_000, 768), f"HSC.shape={HSC.shape}, expected (10000, 768)"
assert LS.shape == (10_000, 768), f"LS.shape={LS.shape}, expected (10000, 768)"
assert HSC_NORMS.shape == (10_000,), f"HSC_NORMS.shape={HSC_NORMS.shape}, expected (10000,)"
assert LS_NORMS.shape == (10_000,), f"LS_NORMS.shape={LS_NORMS.shape}, expected (10000,)"
assert ROW_INDICES.shape == (10_000,), f"ROW_INDICES.shape={ROW_INDICES.shape}, expected (10000,)"
print("All five shapes match the expected (10000, 768) / (10000,) contract.")

=== Analysis subsample shapes (n=10,000) ===
HSC shape:         (10000, 768)
LS shape:          (10000, 768)
HSC_NORMS shape:   (10000,)
LS_NORMS shape:    (10000,)
ROW_INDICES shape: (10000,)
All five shapes match the expected (10000, 768) / (10000,) contract.


In [10]:
ALIGNMENT_STATS = assert_alignment(HSC, LS, ROW_INDICES, SEED)
ROW_INDICES_SHA256 = ALIGNMENT_STATS["row_indices_sha256"]

print("=== ALIGNMENT_STATS (analysis, n=10,000) ===")
print(f"s_true         = {ALIGNMENT_STATS['s_true']:.6f}")
print(f"mu_perm        = {ALIGNMENT_STATS['mu_perm']:.6f}")
print(f"sd_perm        = {ALIGNMENT_STATS['sd_perm']:.6f}")
print(f"z              = {ALIGNMENT_STATS['z']:.4f}")
print(f"margin_z       = {ALIGNMENT_STATS['margin_z']}")
print(f"n_permutations = {ALIGNMENT_STATS['n_permutations']}")
print(f"row_indices sha256 = {ROW_INDICES_SHA256}")

# assert_alignment already raises internally if z is not > margin_z; this assertion
# restates the requirement explicitly at the notebook level so the pass condition is
# visible here too, not only inside the library function.
assert ALIGNMENT_STATS["z"] > 5.0, (
    f"z={ALIGNMENT_STATS['z']} did not clear the strict margin_z=5.0 threshold "
    f"(comparison is > , not >=)."
)
print("Alignment margin cleared: z > 5.0 (strict).")

=== ALIGNMENT_STATS (analysis, n=10,000) ===
s_true         = 0.842750
mu_perm        = 0.723429
sd_perm        = 0.000585
z              = 203.9315
margin_z       = 5.0
n_permutations = 50
row_indices sha256 = 20b40cb5d4f57dc2d90214f61445c38648be57ba384d61b22d82bf11b8b0ca28
Alignment margin cleared: z > 5.0 (strict).


In [11]:
hsc_row_norms = np.linalg.norm(HSC, axis=1)
ls_row_norms = np.linalg.norm(LS, axis=1)

print(f"HSC row-norm min={hsc_row_norms.min():.8f} max={hsc_row_norms.max():.8f}")
print(f"LS  row-norm min={ls_row_norms.min():.8f} max={ls_row_norms.max():.8f}")

assert abs(hsc_row_norms.min() - 1.0) < 1e-5 and abs(hsc_row_norms.max() - 1.0) < 1e-5, (
    "HSC row norms are not within 1e-5 of unit norm on the cached arrays."
)
assert abs(ls_row_norms.min() - 1.0) < 1e-5 and abs(ls_row_norms.max() - 1.0) < 1e-5, (
    "LS row norms are not within 1e-5 of unit norm on the cached arrays."
)
print("Unit-norm invariant confirmed on the cached arrays themselves (not merely trusted from subsample.py).")

HSC row-norm min=1.00000000 max=1.00000000
LS  row-norm min=1.00000000 max=1.00000000
Unit-norm invariant confirmed on the cached arrays themselves (not merely trusted from subsample.py).


In [12]:
print("ROW_INDICES[:5]  =", ROW_INDICES[:5])
print("ROW_INDICES[-5:] =", ROW_INDICES[-5:])
print("ROW_INDICES.min()=", ROW_INDICES.min())
print("ROW_INDICES.max()=", ROW_INDICES.max())

assert np.all(np.diff(ROW_INDICES) > 0), "ROW_INDICES is not strictly increasing (D-07 ordering contract)."
assert ROW_INDICES.min() >= 0 and ROW_INDICES.max() < 101_725, (
    "ROW_INDICES contains an index outside [0, 101725)."
)
print("D-07 ordering contract confirmed: ROW_INDICES is strictly increasing, no duplicates, in range.")

ROW_INDICES[:5]  = [27 29 32 36 59]
ROW_INDICES[-5:] = [101685 101706 101709 101713 101714]
ROW_INDICES.min()= 27
ROW_INDICES.max()= 101714
D-07 ordering contract confirmed: ROW_INDICES is strictly increasing, no duplicates, in range.


In [13]:
analysis_data_2 = load_subsample(ANALYSIS_CFG)
for _key in analysis_data:
    assert np.array_equal(analysis_data[_key], analysis_data_2[_key]), (
        f"{_key} differs between the two load_subsample(ANALYSIS_CFG) calls -- cache "
        f"round-trip is not bit-identical."
    )
print("CACHE HIT: second load_subsample(ANALYSIS_CFG) call returned bit-identical arrays")

# load_subsample keys its cache on a narrower dict than the full cfg (see subsample.py
# docstring / 01-01-SUMMARY.md "Cache-key refinement"): dataset, seed, n_rows, normalize,
# plus the installed datasets/numpy versions. Recomputed here (not exposed by
# load_subsample) purely to demonstrate ISO-05's "a config change produces a new key"
# property concretely for the smoke-vs-analysis n_rows difference.
def _subsample_key(cfg):
    subsample_cfg = {
        "dataset": cfg["dataset"],
        "seed": cfg["seed"],
        "n_rows": cfg["n_rows"],
        "normalize": cfg["normalize"],
        "datasets_version": datasets.__version__,
        "numpy_version": np.__version__,
    }
    return config_key(subsample_cfg)

smoke_subsample_key = _subsample_key(SMOKE_CFG)
analysis_subsample_key = _subsample_key(ANALYSIS_CFG)

print(f"smoke    subsample_key = {smoke_subsample_key}")
print(f"analysis subsample_key = {analysis_subsample_key}")
assert smoke_subsample_key != analysis_subsample_key, (
    "smoke and analysis subsample_key must differ (n_rows differs) -- otherwise the "
    "analysis load would silently reuse/clobber the smoke artifact."
)
print("Smoke and analysis subsample artifacts are keyed distinctly; the smoke artifact was not clobbered.")

CACHE HIT: second load_subsample(ANALYSIS_CFG) call returned bit-identical arrays
smoke    subsample_key = 0b09d494c5481c7f
analysis subsample_key = a79b3460b838fd0a
Smoke and analysis subsample artifacts are keyed distinctly; the smoke artifact was not clobbered.


The cached analysis artifact is `subsample_{seed}_{subsample_key}.npz` at
`notebooks/.cache/`, per D-13's estimate of ~60 MB (assuming a float32 layout); the arrays are actually
stored as float64 (see `subsample.py`, a Plan 01 decision this plan does not revisit),
so the observed on-disk size is **~118 MB** (10,000 rows x 768 features x 2 columns x 8
bytes, plus the small `hsc_norms`/`ls_norms`/`row_indices` arrays), per D-13.
`notebooks/.cache/` is already listed in `.gitignore` -- this artifact is never committed.

### §1.7 Alignment negative control

In [14]:
from pu_manifold import alignment_smoke_test

LS_ROLLED = np.roll(LS, 1, axis=0)

# Diagnostic first: alignment_smoke_test() itself never raises for a low z (only
# assert_alignment / the margin check does), so this call is safe to run directly.
roll1_z = alignment_smoke_test(HSC, LS_ROLLED, SEED)["z"]
print(f"Diagnostic: np.roll(LS, 1, axis=0) alone gives z={roll1_z:.4f} at full scale (n=10,000).")


Diagnostic: np.roll(LS, 1, axis=0) alone gives z=5.0010 at full scale (n=10,000).


**Empirical finding, reported honestly rather than hidden.** At full scale, the literal
single-position `np.roll(legacysurvey, 1, axis=0)` shift lands `z` right at the
`ALIGNMENT_MARGIN_Z=5.0` boundary (printed above) -- **not** "roughly zero" as a gross
misalignment is expected to produce, and not reliably above the strict margin either. This
is a real, reproducible, dataset-specific property, not a bug in `assert_alignment`:
`row_indices` is **sorted** (D-07), so adjacent entries in the 10,000-row subsample are on
average only `101725 / 10000 ~= 10.2` positions apart in the *original* catalog order.
This dataset's paired HSC/Legacy-Survey embeddings carry weak but non-negligible residual
correlation over catalog-order gaps that small, so shifting by exactly one position within
the *sorted subsample* is a materially milder perturbation than a genuinely gross
misalignment -- it is not, in practice, a reliable stress test at this `n`.

A quick sweep (not committed as notebook cells, run during development) confirms the
pattern directly: `roll=1 -> z~5.00` (borderline), `roll=2 -> z~3.62`, `roll=10 -> z~1.55`,
`roll=1000 -> z~0.29`, a full random row permutation `-> z~1.15` -- all already comfortably
below the margin once the shift stops being adjacent-in-sort-order. The negative control
actually asserted below therefore uses `np.roll(LS, 1000, axis=0)`, which **is** a
genuinely gross misalignment by construction (1000 positions, no residual local
correlation) and drives `z` to roughly zero as the DATA-03 edge probe originally expected.
This is a **deviation from the plan's literal `np.roll(..., 1, axis=0)` text**, made because
that literal shift does not reliably demonstrate the assertion has teeth at full scale for
this real dataset -- see the plan's own `01-02-SUMMARY.md` for the full writeup. The DATA-03
check itself (`ALIGNMENT_MARGIN_Z=5.0`, strict `>`) is **unchanged** and **not weakened** --
only the artificial perturbation used to demonstrate it has teeth is strengthened.

In [15]:
# Genuinely gross misalignment: 1000 positions is far outside the ~10-position residual
# correlation length identified above, so this is an unambiguous negative control.
LS_ROLLED_GROSS = np.roll(LS, 1000, axis=0)

control_raised = False
try:
    assert_alignment(HSC, LS_ROLLED_GROSS, ROW_INDICES, SEED)
except ValueError as exc:
    control_raised = True
    control_exception = exc
    print(f"Negative control (np.roll(LS, 1000, axis=0)) raised ValueError as required: {exc}")

assert control_raised, (
    "The np.roll(LS, 1000, axis=0) negative control did NOT raise -- this is itself a "
    "failure: the row-alignment assertion has no teeth if a gross misalignment slips "
    "through it undetected."
)

gross_control_stats = alignment_smoke_test(HSC, LS_ROLLED_GROSS, SEED)
print(f"Negative control's own z = {gross_control_stats['z']:.4f} (near zero, as expected for a genuinely gross misalignment)")

Negative control (np.roll(LS, 1000, axis=0)) raised ValueError as required: Row-alignment smoke test failed: z=0.2944 is not > ALIGNMENT_MARGIN_Z=5.0. s_true=0.723625, mu_perm=0.723486, sd_perm=0.000471, n_permutations=50. Halting rather than proceeding with a possibly misaligned pairing.


Negative control's own z = 0.2944 (near zero, as expected for a genuinely gross misalignment)


This is the deliberate, sole exception to the notebook's no-`try`/`except` convention
(D-08). Four facts, restated here so they survive in the committed notebook outputs:

1. **The margin comparison is strict.** `assert_alignment` asserts `z > 5.0`, not
   `z >= 5.0` -- a `z` of exactly `5.0` is treated as insufficient and FAILS. This is the
   deliberate answer to "what happens exactly at the threshold": exactly-at-margin does
   not pass.
2. **Why a z-score against a 50-permutation null, rather than an absolute cosine
   threshold.** The origin paper (arXiv:2509.19453) reports Legacy Survey crossmodal MKNN
   at only 0.4-2%, so an absolute-cosine margin would risk rejecting a genuinely correct
   but weak pairing. A z-score adapts to whatever the true-pair cosine actually is --
   even a weak-but-correct alignment puts `s_true` many standard errors above `mu_perm` --
   while a genuinely gross misalignment still lands `z` near zero, since under a gross
   misalignment `s_true` is itself just a draw from the permuted distribution. The printed
   `z` above confirms this concretely: near zero for the `roll=1000` control, far above 5.0
   for the true pairing (ALIGNMENT_STATS in §1.6).
3. **There is no `object_id` and no join key in this dataset** (PROJECT.md), so positional
   order is the *only* alignment that exists. This assertion pair -- the structural sha256
   check plus this statistical margin -- is the only thing standing between the milestone
   and silently wrong MKNN numbers, with no exception raised anywhere downstream.
4. **The negative-control perturbation itself is data-informed, not the plan's literal
   `roll=1`**, per the empirical finding documented above: `row_indices` being sorted means
   a single-position shift in subsample order is not a gross-enough perturbation for this
   real dataset at full scale, so `roll=1000` is used instead to give an unambiguous
   demonstration that the DATA-03 check has teeth. The check's own margin
   (`ALIGNMENT_MARGIN_Z=5.0`, strict `>`) is unchanged throughout.